# 04. workflow Pattern — 5 core patterns

## Learning Objectives

Understand Prompt Chaining, Parallelization, Routing, Orchestrator-Worker, and Evaluator-Optimizer patterns.

## 4.1 Environment Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")

In [ ]:
# Observability settings (optional) - LangSmith or Langfuse
# Set the key in .env, or uncomment it below and enter it yourself.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: Automatically activated when LANGSMITH_TRACING=true (no code modification required)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: Pass config={"callbacks": [langfuse_handler]} when calling invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 4.2 Prompt Chaining — Sequential LLM calls

- The output of each step becomes the input of Next Steps
- Purpose: Translation → Verification → Proofreading, Analysis → Summary → Formatting

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class ChainState(TypedDict):
    topic: str
    draft: str
    improved: str

def generate_draft(state: ChainState) -> dict:
    response = model.invoke(f"다음에 대한 한 문장 사실을 작성해주세요: {state['topic']}", config=lf_config)
    return {"draft": response.content}

def improve_draft(state: ChainState) -> dict:
    response = model.invoke(f"다음 텍스트를 더 매력적으로 개선해주세요: {state['draft']}", config=lf_config)
    return {"improved": response.content}

builder = StateGraph(ChainState)
builder.add_node("draft", generate_draft)
builder.add_node("improve", improve_draft)
builder.add_edge(START, "draft")
builder.add_edge("draft", "improve")
builder.add_edge("improve", END)

chain = builder.compile()
result = chain.invoke({"topic": "Python programming"}, config=lf_config)
print(f"초안: {result['draft']}")
print(f"개선: {result['improved']}")

## 4.3 Parallelization — Simultaneous execution of independent `@task`

In [ ]:
from typing import Annotated, TypedDict
import operator


class ParallelState(TypedDict):
    text: str
    analyses: Annotated[list[str], operator.add]


def analyze_tone(state: ParallelState) -> dict:
    r = model.invoke(
        f"한 문장으로 다음의 어조를 설명해주세요: {state['text']}",
        config=lf_config
    )
    return {
        "analyses": [f"어조: {r.content}"]
    }


def analyze_complexity(state: ParallelState) -> dict:
    r = model.invoke(
        f"한 문장으로 다음의 복잡도를 평가해주세요: {state['text']}",
        config=lf_config
    )
    return {
        "analyses": [f"복잡도: {r.content}"]
    }


def synthesize(state: ParallelState) -> dict:
    return {
        "analyses": [
            f"종합: {len(state['analyses'])}개 분석 완료"
        ]
    }


builder = StateGraph(ParallelState)

builder.add_node("tone", analyze_tone)
builder.add_node("complexity", analyze_complexity)
builder.add_node("synthesize", synthesize)

# Parallel branch from START to two analysis nodes
builder.add_edge(START, "tone")
builder.add_edge(START, "complexity")

# Composite after completing two nodes
builder.add_edge("tone", "synthesize")
builder.add_edge("complexity", "synthesize")
builder.add_edge("synthesize", END)

parallel_graph = builder.compile()

result = parallel_graph.invoke(
    {
        "text": "LangGraph enables the powerful agent workflow.",
        "analyses": []
    },
    config=lf_config
)

for a in result["analyses"]:
    print(f"  {a}")

## 4.4 Routing — Classification-based branching

![조건부 라우팅 — classify→technical/business/casual](../assets/images/conditional_routing.png)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, TypedDict


class Classification(BaseModel):
    category: Literal["technical", "business", "casual"]


class RouteState(TypedDict):
    question: str
    category: str
    answer: str


def classify(state: RouteState) -> dict:
    structured = model.with_structured_output(Classification)
    result = structured.invoke(
        f"다음 질문을 분류하세요: {state['question']}",
        config=lf_config
    )
    return {"category": result.category}


def handle_technical(state: RouteState) -> dict:
    r = model.invoke(
        f"기술 전문가로서 답변해주세요: {state['question']}",
        config=lf_config
    )
    return {"answer": r.content}


def handle_business(state: RouteState) -> dict:
    r = model.invoke(
        f"비즈니스 어드바이저로서 답변해주세요: {state['question']}",
        config=lf_config
    )
    return {"answer": r.content}


def handle_casual(state: RouteState) -> dict:
    r = model.invoke(
        f"가볍게 답변해주세요: {state['question']}",
        config=lf_config
    )
    return {"answer": r.content}


def route(state: RouteState) -> str:
    return state["category"]


builder = StateGraph(RouteState)

builder.add_node("classify", classify)
builder.add_node("technical", handle_technical)
builder.add_node("business", handle_business)
builder.add_node("casual", handle_casual)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route)

builder.add_edge("technical", END)
builder.add_edge("business", END)
builder.add_edge("casual", END)

router = builder.compile()

result = router.invoke(
    {
        "question": "How does garbage collection work in Python?"
    },
    config=lf_config
)

print(f"카테고리: {result['category']}")
print(f"답변: {result['answer'][:200]}")

## 4.5 Orchestrator-Worker — Create dynamic worker with Send()

![Orchestrator-Worker 패턴 — plan→Send()→worker(병렬)](../assets/images/orchestrator_worker.png)

In [ ]:
from langgraph.types import Send
from typing import Annotated, TypedDict
import operator


class OrchestratorState(TypedDict):
    topic: str
    sections: list[str]
    results: Annotated[list[str], operator.add]


class WorkerState(TypedDict):
    section: str


def plan_sections(state: OrchestratorState) -> dict:
    r = model.invoke(
        f"'{state['topic']}'에 대한 기사의 짧은 섹션 제목 3개를 나열해주세요. 한 줄에 하나씩, 번호 없이.",
        config=lf_config
    )

    sections = [
        s.strip()
        for s in r.content.strip().split("\n")
        if s.strip()
    ][:3]

    return {"sections": sections}


def assign_workers(state: OrchestratorState) -> list[Send]:
    return [
        Send("worker", {"section": s})
        for s in state["sections"]
    ]


def worker(state: WorkerState) -> dict:
    r = model.invoke(
        f"다음에 대해 한 문장으로 작성해주세요: {state['section']}",
        config=lf_config
    )

    return {
        "results": [
            f"## {state['section']}\n{r.content}"
        ]
    }


builder = StateGraph(OrchestratorState)

builder.add_node("plan", plan_sections)
builder.add_node("worker", worker)

builder.add_edge(START, "plan")

builder.add_conditional_edges(
    "plan",
    assign_workers,
    ["worker"]
)

builder.add_edge("worker", END)

orchestrator = builder.compile()

result = orchestrator.invoke(
    {
        "topic": "machine learning",
        "sections": [],
        "results": []
    },
    config=lf_config
)

for r in result["results"]:
    print(r)
    print()

## 4.6 Evaluator-Optimizer — Generate-evaluation iteration loop

In [ ]:
class EvalState(TypedDict):
    task: str
    draft: str
    feedback: str
    is_good: bool
    iterations: int

def generate(state: EvalState) -> dict:
    if state.get("feedback"):
        prompt = f"피드백을 반영하여 개선해주세요.\n원본: {state['draft']}\n피드백: {state['feedback']}"
    else:
        prompt = f"다음에 대한 한 문장 슬로건을 작성해주세요: {state['task']}"
    r = model.invoke(prompt, config=lf_config)
    return {"draft": r.content, "iterations": state.get("iterations", 0) + 1}

def evaluate(state: EvalState) -> dict:
    r = model.invoke(f"이 슬로건을 1-10으로 평가하고 간단한 피드백을 주세요: '{state['draft']}'", config=lf_config)
    content = r.content
    is_good = any(f"{n}/10" in content for n in range(8, 11))
    return {"feedback": content, "is_good": is_good}

def should_retry(state: EvalState) -> str:
    if state["is_good"] or state["iterations"] >= 3:
        return END
    return "generate"

builder = StateGraph(EvalState)
builder.add_node("generate", generate)
builder.add_node("evaluate", evaluate)

builder.add_edge(START, "generate")
builder.add_edge("generate", "evaluate")
builder.add_conditional_edges("evaluate", should_retry, ["generate", END])

optimizer = builder.compile()
result = optimizer.invoke({"task": "Python learning platform"}, config=lf_config)
print(f"최종 초안 ({result['iterations']}번 반복 후): {result['draft']}")

## 4.7 Pattern comparison table

| pattern | decisive | Parallel | repeat | suitable situation |
|------|--------|------|------|----------|
| Prompt Chaining | O | X | sequential | Step-by-step conversion |
| Parallelization | O | O | X | Independent Analysis |
| Routing | O | X | X | Classification-based processing |
| Orchestrator-Worker | O | O | X | Dynamic Subtasks |
| Evaluator-Optimizer | X | X | O | Quality Improvement Loop |

### Next Steps
→ **[05_agents.ipynb](./05_agents.ipynb)**: Build ReAct agent.